In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Sheet1"
state_col = "state"  # 离散状态列名（整数编码）

df = pd.read_excel(file_path, sheet_name=sheet_name)
states = df[state_col].to_list()

# ========= 2) 参数模板 =========
params = {
    "n_states": 4,       # int: 状态总数（0~n_states-1）
    "n_future": 6,       # int: 预测步数
    "init_state": None   # int/None: 初始状态，None=最后一个历史状态
}

count = np.zeros((params["n_states"], params["n_states"]), dtype=float)
for i in range(len(states) - 1):
    count[states[i], states[i + 1]] += 1

row_sum = count.sum(axis=1, keepdims=True)
P = np.divide(count, np.maximum(row_sum, 1e-12))

cur = states[-1] if params["init_state"] is None else params["init_state"]
pred = []
for _ in range(params["n_future"]):
    nxt = int(np.argmax(P[cur]))
    pred.append(nxt)
    cur = nxt

print("转移矩阵:\n", P)
print("预测状态:", pred)


In [ ]:
"""
马尔可夫预测模型

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "马尔可夫预测模型.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
STATE_COLUMN = "状态"  # TODO: 请填写[状态列名]，说明：该列保存离散状态，不能是连续数值。
FORECAST_STEPS = 3  # TODO: 请填写[预测步数]，说明：正整数，表示向后转移多少期。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # 状态序列必须是一列离散状态，例如 A/B/C 或 1/2/3。
    states = data[STATE_COLUMN].astype(str).tolist()
    unique_states = sorted(set(states))
    index = {s: i for i, s in enumerate(unique_states)}
    matrix = np.zeros((len(unique_states), len(unique_states)))

    # 统计相邻时刻的状态转移次数。
    for a, b in zip(states[:-1], states[1:]):
        matrix[index[a], index[b]] += 1

    # 将次数矩阵按行归一化为转移概率矩阵。
    row_sum = matrix.sum(axis=1, keepdims=True)
    transition = np.divide(matrix, row_sum, out=np.zeros_like(matrix), where=row_sum != 0)

    current = np.zeros(len(unique_states))
    current[index[states[-1]]] = 1
    forecast = current @ np.linalg.matrix_power(transition, FORECAST_STEPS)
    result = pd.DataFrame({"状态": unique_states, f"{FORECAST_STEPS}步后概率": forecast})
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
